# Context Engineering: Windows, Budgets, Memory, and Retrieval Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Token Counter

You cannot budget what you cannot measure. Build a simple token counter (approximation using whitespace splitting, since the exact count depends on the tokenizer).

In [ ]:
```python

import json

import numpy as np

from collections import OrderedDict

def count_tokens(text):

    if not text:

        return 0

    return int(len(text.split()) * 1.3)

def count_tokens_json(obj):

    return count_tokens(json.dumps(obj))

In [ ]:
```

### Step 2: Context Budget Manager

The core abstraction. A budget manager tracks how many tokens each component uses and enforces limits.

In [ ]:
```python

class ContextBudget:

    def __init__(self, max_tokens=128000, generation_reserve=4000):

        self.max_tokens = max_tokens

        self.generation_reserve = generation_reserve

        self.available = max_tokens - generation_reserve

        self.allocations = OrderedDict()

    def allocate(self, component, content, max_tokens=None):

        tokens = count_tokens(content)

        if max_tokens and tokens > max_tokens:

            words = content.split()

            target_words = int(max_tokens / 1.3)

            content = " ".join(words[:target_words])

            tokens = count_tokens(content)

        used = sum(self.allocations.values())

        if used + tokens > self.available:

            allowed = self.available - used

            if allowed <= 0:

                return None, 0

            words = content.split()

            target_words = int(allowed / 1.3)

            content = " ".join(words[:target_words])

            tokens = count_tokens(content)

        self.allocations[component] = tokens

        return content, tokens

    def remaining(self):

        used = sum(self.allocations.values())

        return self.available - used

    def utilization(self):

        used = sum(self.allocations.values())

        return used / self.max_tokens

    def report(self):

        total_used = sum(self.allocations.values())

        lines = []

        lines.append(f"Context Budget Report ({self.max_tokens:,} token window)")

        lines.append("-" * 50)

        for component, tokens in self.allocations.items():

            pct = tokens / self.max_tokens * 100

            bar = "#" * int(pct / 2)

            lines.append(f"  {component:<25} {tokens:>6} tokens ({pct:>5.1f}%) {bar}")

        lines.append("-" * 50)

        lines.append(f"  {'Used':<25} {total_used:>6} tokens ({total_used/self.max_tokens*100:.1f}%)")

        lines.append(f"  {'Generation reserve':<25} {self.generation_reserve:>6} tokens")

        lines.append(f"  {'Remaining':<25} {self.remaining():>6} tokens")

        return "\n".join(lines)

In [ ]:
```

### Step 3: Lost-in-the-Middle Reordering

Implement the reordering strategy: most important items go first and last, least important go in the middle.

In [ ]:
```python

def reorder_lost_in_middle(items, scores):

    paired = sorted(zip(scores, items), reverse=True)

    sorted_items = [item for _, item in paired]

    if len(sorted_items) <= 2:

        return sorted_items

    first_half = sorted_items[::2]

    second_half = sorted_items[1::2]

    second_half.reverse()

    return first_half + second_half

def score_relevance(query, documents):

    query_words = set(query.lower().split())

    scores = []

    for doc in documents:

        doc_words = set(doc.lower().split())

        if not query_words:

            scores.append(0.0)

            continue

        overlap = len(query_words & doc_words) / len(query_words)

        scores.append(round(overlap, 3))

    return scores

In [ ]:
```

### Step 4: Conversation History Compressor

Summarize old conversation turns to reclaim token budget.

In [ ]:
```python

class ConversationManager:

    def __init__(self, max_history_tokens=5000):

        self.turns = []

        self.summaries = []

        self.max_history_tokens = max_history_tokens

    def add_turn(self, role, content):

        self.turns.append({"role": role, "content": content})

        self._compress_if_needed()

    def _compress_if_needed(self):

        total = sum(count_tokens(t["content"]) for t in self.turns)

        if total <= self.max_history_tokens:

            return

        while total > self.max_history_tokens and len(self.turns) > 4:

            old_turns = self.turns[:2]

            summary = self._summarize_turns(old_turns)

            self.summaries.append(summary)

            self.turns = self.turns[2:]

            total = sum(count_tokens(t["content"]) for t in self.turns)

    def _summarize_turns(self, turns):

        parts = []

        for t in turns:

            content = t["content"]

            if len(content) > 100:

                content = content[:100] + "..."

            parts.append(f"{t['role']}: {content}")

        return "Previous: " + " | ".join(parts)

    def get_context(self):

        parts = []

        if self.summaries:

            parts.append("[Conversation Summary]")

            for s in self.summaries:

                parts.append(s)

        parts.append("[Recent Conversation]")

        for t in self.turns:

            parts.append(f"{t['role']}: {t['content']}")

        return "\n".join(parts)

    def token_count(self):

        return count_tokens(self.get_context())

In [ ]:
```

### Step 5: Dynamic Tool Selector

Only include tools relevant to the current query. Classify intent, then filter.

In [ ]:
```python

TOOL_REGISTRY = {

    "read_file": {

        "description": "Read contents of a file",

        "tokens": 120,

        "categories": ["code", "files"],

    },

    "write_file": {

        "description": "Write content to a file",

        "tokens": 150,

        "categories": ["code", "files"],

    },

    "search_code": {

        "description": "Search for patterns in codebase",

        "tokens": 130,

        "categories": ["code"],

    },

    "run_command": {

        "description": "Execute a shell command",

        "tokens": 140,

        "categories": ["code", "system"],

    },

    "create_calendar_event": {

        "description": "Create a new calendar event",

        "tokens": 180,

        "categories": ["calendar"],

    },

    "list_emails": {

        "description": "List recent emails",

        "tokens": 160,

        "categories": ["email"],

    },

    "send_email": {

        "description": "Send an email message",

        "tokens": 200,

        "categories": ["email"],

    },

    "web_search": {

        "description": "Search the web for information",

        "tokens": 140,

        "categories": ["research"],

    },

    "query_database": {

        "description": "Run a SQL query on the database",

        "tokens": 170,

        "categories": ["code", "data"],

    },

    "generate_chart": {

        "description": "Generate a chart from data",

        "tokens": 190,

        "categories": ["data", "visualization"],

    },

}

def classify_intent(query):

    query_lower = query.lower()

    intent_keywords = {

        "code": ["code", "function", "bug", "error", "file", "implement", "refactor", "debug", "test"],

        "calendar": ["meeting", "schedule", "calendar", "appointment", "event"],

        "email": ["email", "mail", "send", "inbox", "message"],

        "research": ["search", "find", "what is", "how does", "explain", "look up"],

        "data": ["data", "query", "database", "chart", "graph", "analytics", "sql"],

    }

    scores = {}

    for intent, keywords in intent_keywords.items():

        score = sum(1 for kw in keywords if kw in query_lower)

        if score > 0:

            scores[intent] = score

    if not scores:

        return ["code"]

    max_score = max(scores.values())

    return [intent for intent, score in scores.items() if score >= max_score * 0.5]

def select_tools(query, token_budget=2000):

    intents = classify_intent(query)

    relevant = {}

    total_tokens = 0

    for name, tool in TOOL_REGISTRY.items():

        if any(cat in intents for cat in tool["categories"]):

            if total_tokens + tool["tokens"] <= token_budget:

                relevant[name] = tool

                total_tokens += tool["tokens"]

    return relevant, total_tokens

In [ ]:
```

### Step 6: Full Context Assembly Pipeline

Wire everything together. Given a query, dynamically assemble the optimal context.

In [ ]:
```python

class ContextEngine:

    def __init__(self, max_tokens=128000, generation_reserve=4000):

        self.budget = ContextBudget(max_tokens, generation_reserve)

        self.conversation = ConversationManager(max_history_tokens=5000)

        self.system_prompt = (

            "You are a helpful AI assistant. You have access to tools for "

            "code editing, file management, web search, and data analysis. "

            "Use the appropriate tools for each task. Be concise and accurate."

        )

        self.knowledge_base = [

            "Python 3.12 introduced type parameter syntax for generic classes using bracket notation.",

            "The project uses PostgreSQL 16 with pgvector for embedding storage.",

            "Authentication is handled by Supabase Auth with JWT tokens.",

            "The frontend is built with Next.js 15 using the App Router.",

            "API rate limits are set to 100 requests per minute per user.",

            "The deployment pipeline uses GitHub Actions with Docker multi-stage builds.",

            "Test coverage must be above 80% for all new modules.",

            "The codebase follows the repository pattern for data access.",

        ]

    def assemble(self, query):

        self.budget = ContextBudget(self.budget.max_tokens, self.budget.generation_reserve)

        system_content, _ = self.budget.allocate("system_prompt", self.system_prompt, max_tokens=1000)

        tools, tool_tokens = select_tools(query, token_budget=2000)

        tool_text = json.dumps(list(tools.keys()))

        tool_content, _ = self.budget.allocate("tools", tool_text, max_tokens=2000)

        relevance = score_relevance(query, self.knowledge_base)

        threshold = 0.1

        relevant_docs = [

            doc for doc, score in zip(self.knowledge_base, relevance)

            if score >= threshold

        ]

        if relevant_docs:

            doc_scores = [s for s in relevance if s >= threshold]

            reordered = reorder_lost_in_middle(relevant_docs, doc_scores)

            doc_text = "\n".join(reordered)

            doc_content, _ = self.budget.allocate("retrieved_context", doc_text, max_tokens=3000)

        history_text = self.conversation.get_context()

        if history_text.strip():

            history_content, _ = self.budget.allocate("conversation_history", history_text, max_tokens=5000)

        query_content, _ = self.budget.allocate("user_query", query, max_tokens=500)

        return self.budget

    def chat(self, query):

        self.conversation.add_turn("user", query)

        budget = self.assemble(query)

        response = f"[Response to: {query[:50]}...]"

        self.conversation.add_turn("assistant", response)

        return budget

def run_demo():

    print("=" * 60)

    print("  Context Engineering Pipeline Demo")

    print("=" * 60)

    engine = ContextEngine(max_tokens=128000, generation_reserve=4000)

    print("\n--- Query 1: Code task ---")

    budget = engine.chat("Fix the bug in the authentication module where JWT tokens expire too early")

    print(budget.report())

    print("\n--- Query 2: Research task ---")

    budget = engine.chat("What is the best approach for implementing vector search in PostgreSQL?")

    print(budget.report())

    print("\n--- Query 3: After conversation history builds up ---")

    for i in range(8):

        engine.conversation.add_turn("user", f"Follow-up question number {i+1} about the implementation details of the system")

        engine.conversation.add_turn("assistant", f"Here is the response to follow-up {i+1} with technical details about the architecture")

    budget = engine.chat("Now implement the changes we discussed")

    print(budget.report())

    print("\n--- Tool Selection Examples ---")

    test_queries = [

        "Fix the bug in auth.py",

        "Schedule a meeting with the team for Tuesday",

        "Show me the database query performance stats",

        "Search for best practices on error handling",

    ]

    for q in test_queries:

        tools, tokens = select_tools(q)

        intents = classify_intent(q)

        print(f"\n  Query: {q}")

        print(f"  Intents: {intents}")

        print(f"  Tools: {list(tools.keys())} ({tokens} tokens)")

    print("\n--- Lost-in-the-Middle Reordering ---")

    docs = ["Doc A (most relevant)", "Doc B (somewhat relevant)", "Doc C (least relevant)",

            "Doc D (relevant)", "Doc E (moderately relevant)"]

    scores = [0.95, 0.60, 0.20, 0.80, 0.50]

    reordered = reorder_lost_in_middle(docs, scores)

    print(f"  Original order: {docs}")

    print(f"  Scores:         {scores}")

    print(f"  Reordered:      {reordered}")

    print(f"  (Most relevant at start and end, least relevant in middle)")

In [ ]:
```

## Exercises

In [ ]:
1. Add a "token waste detector" to the ContextBudget class. It should flag components using more than 30% of the budget and suggest compression strategies specific to each component type (summarize history, prune tools, re-rank documents).

2. Implement semantic deduplication for retrieved context. If two retrieved documents are more than 80% similar (by word overlap or cosine similarity of their embeddings), keep only the higher-scored one. Measure how much token budget this recovers.

3. Build a "context replay" tool. Given a conversation transcript, replay it through the ContextEngine and visualize how the budget allocation changes turn by turn. Plot token usage per component over time. Identify the turn where context starts getting compressed.

4. Implement a priority-based tool selector. Instead of binary include/exclude, assign each tool a relevance score to the current query. Include tools in descending relevance order until the tool budget is exhausted. Compare task performance with 5, 10, 20, and 50 tools included.

5. Build a multi-strategy context compressor. Implement three compression strategies (truncation, summarization, extraction of key sentences) and benchmark them on a set of 20 documents. Measure the tradeoff between compression ratio and information retention (does the compressed version still contain the answer to the query?).